# mHealth: 特徴量テーブルを用いたモデル学習・評価（検証ノート）

このノートは specs/104-mhealth-analysis-trace/tasks.md のタスク3（モデル学習・評価）の実装検証用です。
- 入力: dbt で作成した `featured_activities` テーブル（Athena/Glue Data Catalog）
- 手順: データ取得 → 前処理 → 学習/評価（XGBoost） → メトリクス出力（ローカル/S3 任意）
- 前提: `.env` または `.env.dev` で Athena/Glue/S3 関連の環境変数が設定済み（後続セルで読み込み）


In [2]:
# ライブラリ読み込みと環境変数のロード
from __future__ import annotations
import os, json, time, datetime as dt
import pandas as pd
from dotenv import load_dotenv
import awswrangler as wr
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

# `.env` があれば優先し、なければ `.env.dev` を読む
env_file = os.environ.get('ENV_FILE') or ('.env' if os.path.exists('.env') else '.env.dev')
load_dotenv(env_file)

AWS_REGION = os.getenv('AWS_REGION', 'ap-northeast-1')
ATHENA_WORK_GROUP = os.getenv('ATHENA_WORK_GROUP', 'primary')
S3_STAGING_DIR = os.getenv('S3_STAGING_DIR')  # 例: s3://<bucket>/athena/staging/
DBT_SCHEMA = os.getenv('DBT_SCHEMA', 'dev_processed')  # Athenaのdatabase名に相当
BUCKET = os.getenv('BUCKET')
S3_DATA_DIR = os.getenv('S3_DATA_DIR')  # 例: s3://<bucket>/processed/

print({'env_file': env_file, 'AWS_REGION': AWS_REGION, 'DBT_SCHEMA': DBT_SCHEMA, 'WORK_GROUP': ATHENA_WORK_GROUP})
if S3_STAGING_DIR is None:
    print('[WARN] S3_STAGING_DIR not set. awswrangler will rely on WorkGroup defaults.')
if S3_DATA_DIR is None or BUCKET is None:
    print('[INFO] BUCKET/S3_DATA_DIR not set. S3への結果出力はスキップ可能です。')


{'env_file': '.env.dev', 'AWS_REGION': 'ap-northeast-1', 'DBT_SCHEMA': 'dev_processed', 'WORK_GROUP': 'primary'}
[WARN] S3_STAGING_DIR not set. awswrangler will rely on WorkGroup defaults.
[INFO] BUCKET/S3_DATA_DIR not set. S3への結果出力はスキップ可能です。


## データ取得（Athena: processed.featured_activities）
- AthenaのDatabaseは `DBT_SCHEMA`（例: `dev_processed` / `processed`）を使用
- 失敗時は最小限のダミーデータを生成し、以降の処理を検証可能にする


In [3]:
# Athenaから特徴量テーブルを読み込む
query = "SELECT * FROM featured_activities"
athena_kwargs = dict(database=DBT_SCHEMA, workgroup=ATHENA_WORK_GROUP)
if S3_STAGING_DIR: athena_kwargs['s3_output'] = S3_STAGING_DIR

print({
  'AWS_REGION': AWS_REGION,
  'ATHENA_WORK_GROUP': ATHENA_WORK_GROUP,
  'S3_STAGING_DIR': S3_STAGING_DIR,
  'DBT_SCHEMA': DBT_SCHEMA,
  'BUCKET': BUCKET,
  'S3_DATA_DIR': S3_DATA_DIR
})

df = None
try:
    df = wr.athena.read_sql_query(sql=query, **athena_kwargs)
    print(f'Read {len(df):,} rows from {DBT_SCHEMA}.featured_activities')
    display(df.head())
    df.shape
except Exception as e:
    print('[WARN] Athena read failed; falling back to synthetic sample. Reason:', e)




{'AWS_REGION': 'ap-northeast-1', 'ATHENA_WORK_GROUP': 'primary', 'S3_STAGING_DIR': None, 'DBT_SCHEMA': 'dev_processed', 'BUCKET': None, 'S3_DATA_DIR': None}
Read 120 rows from dev_processed.featured_activities


,user_id,activity_label,chest_acc_mean,chest_acc_std,chest_acc_min,chest_acc_max,left_ankle_acc_mean,left_ankle_acc_std,left_ankle_acc_min,left_ankle_acc_max,right_lower_arm_acc_mean,right_lower_arm_acc_std,right_lower_arm_acc_min,right_lower_arm_acc_max
0,2,12,-2.908749,3.865843,-13.523100,5.163660,-3.533463,4.449494,-19.534333,11.074300,-2.992116,4.527436,-14.203033,6.836467
1,7,12,-2.716955,5.582261,-19.927667,7.493007,-3.212388,4.143294,-16.625700,9.008900,-1.517064,3.351027,-9.767800,8.510700
2,9,12,-2.604597,5.305858,-19.910000,6.884367,-3.419034,4.785406,-20.208000,10.742306,-1.480875,4.095683,-14.171800,8.169587
3,3,1,-3.343736,0.140739,-4.008687,-2.699743,-2.896595,0.177095,-4.030837,-2.484067,-3.501719,0.129646,-4.453867,-2.755901
4,5,10,-3.731168,4.488633,-18.115333,3.463600,-4.524628,3.438479,-20.218333,17.425333,-3.167607,3.895766,-15.307900,4.512733


In [4]:
# Fallback: df が未取得/空のときは合成データを生成
if df is None or len(df) == 0:
    print('[INFO] Using synthetic fallback dataset.')
    rng = np.random.default_rng(42)
    n_users, n_classes, n_rows = 6, 4, 120
    users = rng.integers(1, n_users + 1, size=n_rows)
    labels = rng.integers(1, n_classes + 1, size=n_rows)
    df = pd.DataFrame({
        'user_id': users,
        'activity_label': labels,
        'feat_1': rng.normal(size=n_rows) + labels * 0.1,
        'feat_2': rng.normal(size=n_rows) + users * 0.05,
        'feat_3': rng.normal(size=n_rows)
    })
    print(f'[INFO] Generated synthetic sample: {len(df)} rows, users={sorted(df.user_id.unique())}, labels={sorted(df.activity_label.unique())}')


## 前処理と分割戦略
- 目的変数: `activity_label`
- 特徴量: `user_id` と `activity_label` 以外の全列
- 分割: 被験者リークを避けるため `user_id` をグループとして GroupShuffleSplit で学習/評価を分離（再現性のため固定seed）
  - 参考: タスク3のスクリプト実装では同等のオプションを設ける想定


In [5]:
# 特徴量・目的変数の抽出（ラベルは0始まりにエンコード）
target_col = 'activity_label'
drop_cols = ['user_id', target_col]
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols].copy()
y_raw = df[target_col].astype(int).copy()
groups = df['user_id'].astype(str).copy()

le = LabelEncoder()
y = pd.Series(le.fit_transform(y_raw), index=y_raw.index)
label_map = {int(orig): int(enc) for enc, orig in enumerate(le.classes_.tolist())}
print('Features:', feature_cols)
print('Classes (raw->enc):', label_map)
print('Class counts (raw):', y_raw.value_counts().sort_index().to_dict())

# Group-based split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_enc, y_test_enc = y.iloc[train_idx], y.iloc[test_idx]
print(f'Train/Test shapes: {X_train.shape} / {X_test.shape}')
print('Train users:', sorted(df.iloc[train_idx]['user_id'].unique()))
print('Test users:', sorted(df.iloc[test_idx]['user_id'].unique()))


Features: ['chest_acc_mean', 'chest_acc_std', 'chest_acc_min', 'chest_acc_max', 'left_ankle_acc_mean', 'left_ankle_acc_std', 'left_ankle_acc_min', 'left_ankle_acc_max', 'right_lower_arm_acc_mean', 'right_lower_arm_acc_std', 'right_lower_arm_acc_min', 'right_lower_arm_acc_max']
Classes (raw->enc): {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 11: 10, 12: 11}
Class counts (raw): {1: 10, 2: 10, 3: 10, 4: 10, 5: 10, 6: 10, 7: 10, 8: 10, 9: 10, 10: 10, 11: 10, 12: 10}
Train/Test shapes: (96, 12) / (24, 12)
Train users: ['1', '2', '3', '4', '5', '6', '7', '9']
Test users: ['10', '8']


## XGBoost による学習・評価
- 少数特徴量のため軽量なパラメータで学習
- 評価指標: Accuracy, Macro F1


In [6]:
# モデル学習（エンコード済みラベルで学習）
clf = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train_enc)

# 予測・評価（エンコード済みで評価し、レポートは元ラベルで表示）
y_pred_enc = clf.predict(X_test)
acc = accuracy_score(y_test_enc, y_pred_enc)
f1 = f1_score(y_test_enc, y_pred_enc, average='macro')
print({'accuracy': round(acc, 4), 'macro_f1': round(f1, 4)})
# 元ラベルへ逆変換
y_pred = pd.Series(le.inverse_transform(y_pred_enc.astype(int)))
y_test_raw = y_raw.iloc[test_idx].reset_index(drop=True)
print('Classification Report\n', classification_report(y_test_raw, y_pred, digits=4))


{'accuracy': 0.75, 'macro_f1': 0.7111}
Classification Report
               precision    recall  f1-score   support

           1     0.6667    1.0000    0.8000         2
           2     1.0000    1.0000    1.0000         2
           3     1.0000    0.5000    0.6667         2
           4     0.6667    1.0000    0.8000         2
           5     1.0000    0.5000    0.6667         2
           6     0.6667    1.0000    0.8000         2
           7     1.0000    1.0000    1.0000         2
           8     0.6667    1.0000    0.8000         2
           9     0.0000    0.0000    0.0000         2
          10     0.5000    1.0000    0.6667         2
          11     1.0000    0.5000    0.6667         2
          12     1.0000    0.5000    0.6667         2

    accuracy                         0.7500        24
   macro avg     0.7639    0.7500    0.7111        24
weighted avg     0.7639    0.7500    0.7111        24



/home/riku_nishikawa/dev/data-flow/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/riku_nishikawa/dev/data-flow/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/riku_nishikawa/dev/data-flow/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

## 評価結果の保存（任意でS3へ出力）
- `BUCKET`/`S3_DATA_DIR` が設定されていれば、`processed/model_results/` 配下へJSONで出力
- タスク3スクリプト（`model_training/train_evaluate.py`）では、この仕様を踏襲しS3へ保存する


In [7]:
# ローカル出力 + 任意のS3出力
metrics = {
    'timestamp': dt.datetime.utcnow().isoformat()+'Z',
    'dbt_schema': DBT_SCHEMA,
    'rows': int(len(df)),
    'features': feature_cols,
    'accuracy': float(acc),
    'macro_f1': float(f1),
    'train_users': sorted(set(df.iloc[train_idx]['user_id'].astype(str))),
    'test_users': sorted(set(df.iloc[test_idx]['user_id'].astype(str))),
}

# ローカルに保存
os.makedirs('artifacts', exist_ok=True)
local_path = f'artifacts/mhealth_metrics_{int(time.time())}.json'
with open(local_path, 'w') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Saved local:', local_path)

# S3へ保存（設定がある場合のみ）
if S3_DATA_DIR and S3_DATA_DIR.startswith('s3://'):
    key = f"model_results/mhealth_metrics_{int(time.time())}.json"
    s3_uri = S3_DATA_DIR.rstrip('/') + '/' + key
    # awswranglerはJSON直接書き込みユーティリティがないため、boto3でアップロード
    import boto3, io
    s3 = boto3.client('s3', region_name=AWS_REGION)
    bucket = S3_DATA_DIR.replace('s3://', '').split('/')[0]
    prefix = '/'.join(S3_DATA_DIR.replace('s3://', '').split('/')[1:] + [key]) if '/' in S3_DATA_DIR.replace('s3://', '') else key
    body = json.dumps(metrics, ensure_ascii=False).encode('utf-8')
    s3.put_object(Bucket=bucket, Key=prefix, Body=body, ContentType='application/json')
    print('Uploaded to S3:', f's3://{bucket}/'+prefix)
else:
    print('[INFO] S3_DATA_DIR is not set. Skipped uploading metrics to S3.')


Saved local: artifacts/mhealth_metrics_1758453569.json
[INFO] S3_DATA_DIR is not set. Skipped uploading metrics to S3.


## 次のアクション
- 本ノートのロジックを `model_training/train_evaluate.py` に移植（タスク3）
- 引数/環境変数で `DBT_SCHEMA`、`S3_DATA_DIR`、分割戦略（Group/Random）を切替可能にする
- Lambda実行時は Layer に `awswrangler`, `xgboost`, `scikit-learn` を含める（タスク4）
